# 07 · Does any of this generalize past the Front Range?

Every finding so far comes from one semi-arid Colorado landscape where elevation, land cover,
and development happen to be tightly confounded (mountains = forested + undeveloped, plains =
dry + built up). That confound is exactly why notebook 04 couldn't fully separate "Clay senses
terrain" from "Clay senses terrain's downstream effects here." This notebook re-runs the same
pipeline -- fetch, chip, embed, PCA, cluster-vs-WorldCover, elevation correlation -- on a
deliberately different landscape: **Seattle / Puget Sound, WA**. Marine climate, year-round
evergreen forest instead of a strongly seasonal one, real open water at scale (Puget Sound +
Lake Washington, not a single reservoir), a denser urban core, and a much smaller elevation range
within the AOI than the Front Range's ~1,800m of relief.

Three specific questions, each checked against the Front Range numbers already in the paper:

1. Is PC1 still dominated by one axis (~59% of variance there), and does it still correlate more
   with elevation than with NDVI/NDBI the way it did in Colorado (r = -0.82 vs -0.55/0.61)?
2. Do unsupervised clusters still agree with independent ESA WorldCover land cover at a similar
   level (ARI = 0.275 in Colorado), and does a genuine open-water cluster emerge here, given
   Puget Sound and Lake Washington are far larger than anything in the Front Range AOI?
3. Does the elevation confound weaken when the AOI's actual elevation range is much smaller (a
   few hundred meters here vs. ~1,800m there)?

**Requires a GPU runtime.** Single self-contained notebook -- the AOI is small enough (~200
chips) that fetch, chip, embed, and analysis all fit in one sitting, no Drive handoff needed.

In [ ]:
REPO_URL = "https://github.com/ZanderHirman08/SATEMB.git"

import os

if not os.path.exists("SATEMB"):
    !git clone {REPO_URL}
%cd SATEMB
!pip install -q -r environment/requirements-colab.txt

In [ ]:
import sys

sys.path.append(os.getcwd())

import matplotlib.pyplot as plt
import numpy as np
import odc.stac
import pandas as pd
import torch
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score

from src import clay_embed, stac_utils, viz_utils

assert torch.cuda.is_available(), "No GPU detected -- switch runtime type to T4 GPU and re-run"
device = "cuda"
os.makedirs("docs/figures", exist_ok=True)

catalog = stac_utils.open_catalog()
BBOX = stac_utils.SEATTLE_BBOX
CRS = stac_utils.SEATTLE_CRS
print(f"AOI: {BBOX}, CRS: {CRS}")

## Fetch Seattle imagery

Same summer window as the Front Range study (Jun-Sep 2024) for a fair comparison, same
per-tile least-cloudy selection in case the AOI spans more than one Sentinel-2 tile.

In [ ]:
items_all = stac_utils.search_sentinel2(catalog, bbox=BBOX, datetime_range="2024-06-01/2024-09-15", max_cloud_cover=15.0)
print(f"{len(items_all)} candidate scenes")
mosaic_items = stac_utils.select_least_cloudy_per_tile(items_all)

ds = odc.stac.load(
    mosaic_items, bands=stac_utils.S2_BANDS, bbox=BBOX, crs=CRS,
    resolution=stac_utils.GSD_M, groupby="solar_day", chunks={"x": 1024, "y": 1024},
)
mosaic = ds.to_array(dim="band").median(dim="time").compute()
print(mosaic.shape, mosaic.dtype)

In [ ]:
rgb = mosaic.sel(band=["B04", "B03", "B02"]).values.astype("float32")
out = np.zeros_like(rgb)
for i in range(3):
    lo, hi = np.nanpercentile(rgb[i], [2, 98])
    out[i] = np.clip((rgb[i] - lo) / (hi - lo + 1e-6), 0, 1)

plt.figure(figsize=(10, 10))
plt.imshow(out.transpose(1, 2, 0))
plt.title(f"Seattle / Puget Sound mosaic -- {mosaic.sizes['y']}x{mosaic.sizes['x']} px")
plt.axis("off")
plt.savefig("docs/figures/seattle_true_color.png", dpi=150, bbox_inches="tight")
plt.show()

## Chip and embed

In [ ]:
height, width = mosaic.sizes["y"], mosaic.sizes["x"]
grid = stac_utils.make_pixel_chip_grid(height, width)
print(f"{len(grid)} candidate chips ({height // stac_utils.CHIP_SIZE_PX} rows x {width // stac_utils.CHIP_SIZE_PX} cols)")

x_coords, y_coords = mosaic.x.values, mosaic.y.values
raster_crs = ds.odc.crs
acquisition_date = str(mosaic_items[0].datetime.date())
NODATA_FRAC_THRESHOLD = 0.05

chips_meta, chip_pixels = [], []
for chip in grid:
    arr = mosaic.values[:, chip["y_slice"], chip["x_slice"]]
    if np.isnan(arr).mean() > NODATA_FRAC_THRESHOLD:
        continue
    bounds = stac_utils.pixel_window_to_lonlat_bounds(x_coords, y_coords, chip, raster_crs)
    lat, lon = stac_utils.bounds_centroid(bounds)
    chips_meta.append({"id": chip["id"], "bounds": bounds, "lat": lat, "lon": lon, "date": acquisition_date})
    chip_pixels.append(np.nan_to_num(arr, nan=0.0).astype("float32"))

chip_pixels = np.stack(chip_pixels)
print(f"Kept {len(chips_meta)}/{len(grid)} chips -> {chip_pixels.shape}, {chip_pixels.nbytes / 1e9:.2f} GB")

In [ ]:
ckpt_path = clay_embed.download_checkpoint()
metadata_path = clay_embed.download_metadata_yaml()
model = clay_embed.load_model(ckpt_path, metadata_path, device=device)
wavelengths, band_means, band_stds = clay_embed.load_band_stats(metadata_path)

BATCH_SIZE = 16
dates = [pd.Timestamp(m["date"]) for m in chips_meta]
lats = [m["lat"] for m in chips_meta]
lons = [m["lon"] for m in chips_meta]

all_embeddings = []
n = chip_pixels.shape[0]
for start in range(0, n, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n)
    batch_pixels = clay_embed.normalize_chips(chip_pixels[start:end], band_means, band_stds)
    time_feats, latlon_feats = clay_embed.make_time_latlon_tensors(dates[start:end], lats[start:end], lons[start:end])
    batch_emb = clay_embed.encode_batch(model, batch_pixels, time_feats, latlon_feats, wavelengths, device=device)
    all_embeddings.append(batch_emb)
    del batch_pixels, time_feats, latlon_feats
    torch.cuda.empty_cache()
    if (start // BATCH_SIZE) % 5 == 0:
        print(f"  encoded {end}/{n}")

embeddings = np.concatenate(all_embeddings, axis=0)
print(f"Final embeddings: {embeddings.shape}")

## PCA structure -- compare to the Front Range

Front Range reference (from the paper): PC1 explains 59.4% of variance, r=-0.82 with elevation,
r=0.61 with NDBI, r=-0.55 with NDVI.

In [ ]:
BLUE, GREEN, RED, NIR, SWIR1 = 0, 1, 2, 6, 8

def mean_index(pixels, band_a, band_b):
    a = pixels[:, band_a].mean(axis=(1, 2))
    b = pixels[:, band_b].mean(axis=(1, 2))
    return (a - b) / (a + b + 1e-6)

ndvi = mean_index(chip_pixels, NIR, RED)
ndbi = mean_index(chip_pixels, SWIR1, NIR)
ndwi = mean_index(chip_pixels, GREEN, NIR)

N_COMPONENTS = 10
pca = PCA(n_components=N_COMPONENTS, random_state=0).fit(embeddings)
pcs = pca.transform(embeddings)

print(f"{'PC':4s} {'var%':>6s} {'r(NDVI)':>9s} {'r(NDBI)':>9s} {'r(NDWI)':>9s}")
for i in range(N_COMPONENTS):
    r_ndvi = stats.pearsonr(pcs[:, i], ndvi)[0]
    r_ndbi = stats.pearsonr(pcs[:, i], ndbi)[0]
    r_ndwi = stats.pearsonr(pcs[:, i], ndwi)[0]
    var_pct = pca.explained_variance_ratio_[i] * 100
    print(f"PC{i+1:<3d}{var_pct:6.1f}% {r_ndvi:9.2f} {r_ndbi:9.2f} {r_ndwi:9.2f}")

print(f"\nFront Range reference: PC1=59.4% var, r(NDVI)=-0.55, r(NDBI)=0.61")

In [ ]:
pca_colors, explained_variance = viz_utils.pca_to_hex_colors(embeddings)

rows = [int(m["id"][1:4]) for m in chips_meta]
cols = [int(m["id"][5:8]) for m in chips_meta]
n_rows, n_cols = max(rows) + 1, max(cols) + 1

preview = np.zeros((n_rows, n_cols, 3), dtype=np.uint8)
for r, c, hexcolor in zip(rows, cols, pca_colors):
    preview[r, c] = [int(hexcolor[i : i + 2], 16) for i in (1, 3, 5)]

plt.figure(figsize=(10, 10))
plt.imshow(preview)
plt.title("Seattle: Clay embeddings, PCA-projected to RGB (top 3 components)")
plt.axis("off")
plt.savefig("docs/figures/seattle_pca_semantic_map.png", dpi=150, bbox_inches="tight")
plt.show()

## Cluster and validate against ESA WorldCover

Front Range reference: Adjusted Rand Index = 0.275, one cluster ~pure Built-up, one ~pure Tree
cover, water too sparse to form its own cluster.

In [ ]:
clusters = viz_utils.cluster_embeddings(embeddings, method="kmeans", n_clusters=8)
print(f"Cluster sizes: {np.bincount(clusters)}")

WORLDCOVER_CLASSES = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare/sparse", 70: "Snow/ice", 80: "Water",
    90: "Wetland", 95: "Mangroves", 100: "Moss/lichen",
}

wc_items = stac_utils.search_worldcover(catalog, bbox=BBOX)
wc_ds = odc.stac.load(wc_items, bbox=BBOX, crs=CRS, resolution=stac_utils.GSD_M, chunks={"x": 1024, "y": 1024})
wc_var = wc_ds["map"]
wc_map = wc_var.isel(time=0).compute().values if "time" in wc_var.dims else wc_var.compute().values

wc_grid = {c["id"]: c for c in stac_utils.make_pixel_chip_grid(*wc_map.shape)}

majority_class, matched_clusters = [], []
for meta, cluster in zip(chips_meta, clusters):
    win = wc_grid.get(meta["id"])
    if win is None:
        continue
    patch = wc_map[win["y_slice"], win["x_slice"]]
    values, counts = np.unique(patch[patch > 0], return_counts=True)
    if len(values) == 0:
        continue
    majority_class.append(values[np.argmax(counts)])
    matched_clusters.append(cluster)

print(f"Matched {len(majority_class)}/{len(chips_meta)} chips to a WorldCover majority class")
ari = adjusted_rand_score(majority_class, matched_clusters)
print(f"Adjusted Rand Index (Seattle): {ari:.3f}   (Front Range reference: 0.275)")

class_names = [WORLDCOVER_CLASSES.get(c, str(c)) for c in majority_class]
contingency = pd.crosstab(pd.Series(matched_clusters, name="embedding cluster"), pd.Series(class_names, name="WorldCover class"))
print(contingency)

In [ ]:
plt.figure(figsize=(10, 6))
plt.imshow(contingency.values, aspect="auto", cmap="viridis")
plt.xticks(range(len(contingency.columns)), contingency.columns, rotation=45, ha="right")
plt.yticks(range(len(contingency.index)), contingency.index)
plt.xlabel("ESA WorldCover class")
plt.ylabel("Embedding cluster")
plt.title(f"Seattle: embedding clusters vs. WorldCover (ARI={ari:.3f})")
plt.colorbar(label="# chips")
plt.tight_layout()
plt.savefig("docs/figures/seattle_cluster_vs_worldcover.png", dpi=150)
plt.show()

## Elevation check

Front Range reference: PC1 vs. elevation r=-0.82, over an AOI spanning ~1,800m of relief. This
AOI's elevation range is much smaller -- worth checking whether the correlation weakens
proportionally.

In [ ]:
dem_items = stac_utils.search_dem(catalog, bbox=BBOX)
dem_ds = odc.stac.load(dem_items, bands=["data"], bbox=BBOX, crs=CRS, resolution=stac_utils.GSD_M, chunks={"x": 1024, "y": 1024})
dem_var = dem_ds["data"]
elevation_raster = dem_var.median(dim="time").compute().values if "time" in dem_var.dims else dem_var.compute().values
elevation_raster = np.where(elevation_raster < -1000, np.nan, elevation_raster)

dem_grid = {c["id"]: c for c in stac_utils.make_pixel_chip_grid(*elevation_raster.shape)}
mean_elev = []
for meta in chips_meta:
    win = dem_grid.get(meta["id"])
    mean_elev.append(float(np.nanmean(elevation_raster[win["y_slice"], win["x_slice"]])) if win else np.nan)
mean_elev = np.array(mean_elev)

valid = ~np.isnan(mean_elev)
print(f"Elevation range in this AOI: {np.nanmin(mean_elev):.0f}-{np.nanmax(mean_elev):.0f} m "
      f"(Front Range reference: 1,455-3,253 m)")
r_elev_pc1 = stats.pearsonr(pcs[valid, 0], mean_elev[valid])
print(f"PC1 vs. elevation (Seattle): r={r_elev_pc1[0]:.3f}, p={r_elev_pc1[1]:.4f}   (Front Range reference: r=-0.82)")

## Summary: side-by-side with the Front Range

In [ ]:
print(f"{'metric':38s} {'Front Range':>14s} {'Seattle':>14s}")
print(f"{'chips':38s} {725:>14d} {len(chips_meta):>14d}")
print(f"{'PC1 variance explained':38s} {'59.4%':>14s} {pca.explained_variance_ratio_[0]*100:>13.1f}%")
print(f"{'PC1 vs. elevation (r)':38s} {-0.82:>14.2f} {r_elev_pc1[0]:>14.2f}")
print(f"{'Adjusted Rand Index vs. WorldCover':38s} {0.275:>14.3f} {ari:>14.3f}")
print(f"{'AOI elevation range (m)':38s} {'1,455-3,253':>14s} {f'{np.nanmin(mean_elev):.0f}-{np.nanmax(mean_elev):.0f}':>14s}")
print()
print("Done. Commit the new docs/figures/seattle_*.png files back to the repo.")
print("(This notebook doesn't touch docs/data/chips.geojson -- it's a separate AOI/region.)")